In [2]:
print("Hello India")

Hello India


1. scatter plots - visualize the relationship between two quantitative variables

2. line plots - visualize trends with respect to an independent, ordered quantity (e.g., time)

3. bar plots visualize comparisons of amounts

4. histograms visualize the distribution of one quantitative variable (i.e., all its possible values and how often they occur)



Question: Does the concentration of atmospheric CO2
change over time, and are there any interesting patterns to note?

In [3]:
import pandas as pd
import altair as alt

In [4]:
co2_df = pd.read_csv("data/mauna_loa_data.csv")
co2_df

,date_measured,ppm
0,1980-02-01,338.34
1,1980-03-01,340.01
2,1980-04-01,340.93
3,1980-05-01,341.48
4,1980-06-01,341.33
...,...,...
479,2020-02-01,414.11
480,2020-03-01,414.51
481,2020-04-01,416.21
482,2020-05-01,417.07


In [5]:
co2_df = pd.read_csv("data/mauna_loa_data.csv",
                     parse_dates=["date_measured"])
co2_df

,date_measured,ppm
0,1980-02-01,338.34
1,1980-03-01,340.01
2,1980-04-01,340.93
3,1980-05-01,341.48
4,1980-06-01,341.33
...,...,...
479,2020-02-01,414.11
480,2020-03-01,414.51
481,2020-04-01,416.21
482,2020-05-01,417.07


In [6]:
co2_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date_measured  484 non-null    datetime64[ns]
 1   ppm            484 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 7.7 KB


Since we are investigating a relationship between two variables (CO
 concentration and date), a scatter plot is a good place to start. Scatter plots show the data as individual points with x (horizontal axis) and y (vertical axis) coordinates. Here, we will use the measurement date as the x coordinate and the CO
 concentration as the y coordinate. We create a chart with the alt.Chart() function.

There are a few basic aspects of a plot that we need to specify:

The name of the data frame to visualize.

Here, we specify the co2_df data frame as an argument to alt.Chart

The graphical mark, which specifies how the mapped data should be displayed.

To create a graphical mark, we use Chart.mark_* methods (see the altair reference for a list of graphical mark).

Here, we use the mark_point function to visualize our data as a scatter plot.

The encoding channels, which tells altair how the columns in the data frame map to visual properties in the chart.

To create an encoding, we use the encode function.

The encode method builds a key-value mapping between encoding channels (such as x, y) to fields in the data set, accessed by field name (column names)

Here, we set the x axis of the plot to the date_measured variable, and on the y axis, we plot the ppm variable.

For the y-axis, we also provided the method scale(zero=False). By default, altair chooses the y-limits based on the data and will keep y=0 in view. This is often a helpful default, but here it makes it difficult to see any trends in our data since the smallest value is >300 ppm. So by providing scale(zero=False), we tell altair to choose a reasonable lower bound based on our data, and that lower bound doesn’t have to be zero.

To change the properties of the encoding channels, we need to leverage the helper functions alt.Y and alt.X. These helpers have the role of customizing things like order, titles, and scales. Here, we use alt.Y to change the domain of the y-axis, so that it starts from the lowest value in the date_measured column rather than from zero.



In [7]:
co2_scatter = alt.Chart(co2_df).mark_point().encode(
    x="date_measured",
    y=alt.Y("ppm").scale(zero=False)
)

co2_scatter

alt.Chart(...)

We can create a line plot in altair using the mark_line function. Let’s now try to visualize the co2_df as a line plot with just the default arguments:

In [8]:
co2_line = alt.Chart(co2_df).mark_line().encode(
    x="date_measured",
    y=alt.Y("ppm").scale(zero=False)
)
co2_line

alt.Chart(...)

To add axis labels, we use the title method along with alt.X and alt.Y functions. To change the font size, we use the configure_axis function with the titleFontSize argument.

In [9]:
co2_line = alt.Chart(co2_df).mark_line().encode(
    x=alt.X("date_measured").title("Year"),
    y=alt.Y("ppm").scale(zero=False).title("Atmospheric CO2 (ppm)")
).configure_axis(titleFontSize=12)
co2_line

alt.Chart(...)

it is totally fine to use a small number of visualizations to answer different aspects of the question you are trying to answer. We will accomplish this by using scale, another important feature of altair that easily transforms the different variables and set limits. In particular, here, we will use the alt.Scale function to zoom in on just a few years of data (say, 1990-1995). The domain argument takes a list of length two to specify the upper and lower bounds to limit the axis. We also added the argument clip=True to mark_line. This tells altair to “clip” (remove) the data outside of the specified domain that we set so that it doesn’t extend past the plot area. Since we are using both the scale and title method on the encodings we stack them on separate lines to make the code easier to read.

In [10]:
co2_line_scale = alt.Chart(co2_df).mark_line(clip=True).encode(
    x=alt.X("date_measured")
    .scale(domain=["1990", "1995"]).title("Measurement Date"),
    y=alt.Y("ppm").scale(zero=False).title("Atmospheric CO2 (ppm)")
).configure_axis(titleFontSize=12)

co2_line_scale

alt.Chart(...)

Scatter plots: the Old Faithful eruption time data set

The faithful data set contains measurements of the waiting time between eruptions and the subsequent eruption duration (in minutes) of the Old Faithful geyser in Yellowstone National Park, Wyoming, United States. First, we will read the data and then answer the following question:

Question: Is there a relationship between the waiting time before an eruption and the duration of the eruption?

In [11]:
faithful = pd.read_csv("data/faithful.csv")
faithful

,eruptions,waiting
0,3.600,79.0
1,1.800,54.0
2,3.333,74.0
3,2.283,62.0
4,4.533,85.0
...,...,...
267,4.117,81.0
268,2.150,46.0
269,4.417,90.0
270,1.817,46.0


we investigate the relationship between two quantitative variables (waiting time and eruption time). But if you look at the output of the data frame, you’ll notice that unlike time in the Mauna Loa CO
 data set, neither of the variables here have a natural order to them. So a scatter plot is likely to be the most appropriate visualization. Let’s create a scatter plot using the altair package with the waiting variable on the horizontal axis, the eruptions variable on the vertical axis, and mark_point as the graphical mark.

In [12]:
faithful_scatter = alt.Chart(faithful).mark_point().encode(
    x="waiting",
    y="eruptions"
)

faithful_scatter

alt.Chart(...)

We can see in Fig. 4.6 that the data tend to fall into two groups: one with short waiting and eruption times, and one with long waiting and eruption times. Note that in this case, there is no overplotting: the points are generally nicely visually separated, and the pattern they form is clear. In order to refine the visualization, we need only to add axis labels and make the font more readable.



In [13]:
faithful_scatter_labels = alt.Chart(faithful).mark_point().encode(
    x=alt.X("waiting").title("Waiting Time (mins)"),
    y=alt.Y("eruptions").title("Eruption Duration (mins)")
)

faithful_scatter_labels

alt.Chart(...)

We can change the size of the point and color of the plot by specifying mark_point(size=10, color="black").

In [14]:
faithful_scatter_labels_black = alt.Chart(faithful).mark_point(size=10, color="black").encode(
    x=alt.X("waiting").title("Waiting Time (mins)"),
    y=alt.Y("eruptions").title("Eruption Duration (mins)")
)

faithful_scatter_labels_black

alt.Chart(...)

Recall the can_lang data set. It contains counts of languages from the 2016 Canadian census.

Question: Is there a relationship between the percentage of people who speak a language as their mother tongue and the percentage for whom that is the primary language spoken at home? And is there a pattern in the strength of this relationship in the higher-level language categories (Official languages, Aboriginal languages, or non-official and non-Aboriginal languages)?

In [15]:
can_lang = pd.read_csv("data/can_lang.csv")
can_lang

,category,language,mother_tongue,most_at_home,most_at_work,lang_known
0,Aboriginal languages,"Aboriginal languages, n.o.s.",590,235,30,665
1,Non-Official & Non-Aboriginal languages,Afrikaans,10260,4785,85,23415
2,Non-Official & Non-Aboriginal languages,"Afro-Asiatic languages, n.i.e.",1150,445,10,2775
3,Non-Official & Non-Aboriginal languages,Akan (Twi),13460,5985,25,22150
4,Non-Official & Non-Aboriginal languages,Albanian,26895,13135,345,31930
...,...,...,...,...,...,...
209,Non-Official & Non-Aboriginal languages,Wolof,3990,1385,10,8240
210,Aboriginal languages,Woods Cree,1840,800,75,2665
211,Non-Official & Non-Aboriginal languages,Wu (Shanghainese),12915,7650,105,16530
212,Non-Official & Non-Aboriginal languages,Yiddish,13555,7085,895,20985
